In [1]:
%load_ext autotime

import glob
import pandas as pd
import numpy as np
import plotly.express as px
import pyperclip


def df_to_clipboard(df, index=False):
    pyperclip.copy(df.to_markdown(index=index))

# Load files

In [2]:
files = glob.glob("data/results_*.jsonl")
files

['data/results_refugee.jsonl',
 'data/results_unhcr.jsonl',
 'data/results_prwp.jsonl']

In [3]:
data = pd.concat([pd.read_json(x, lines=True) for x in files]).reset_index(drop=True)
data

,snapshot_id,source,model,raw_output,usage,cost,status,incomplete_details,parsed_output,error
0,027_Jordan-Emergency-Food-Security-Project_fig...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""figure_title"",""o...","{'input_tokens': 7168, 'input_tokens_details':...","{'input_tokens': 7168, 'output_tokens': 2431, ...",completed,NaN,"{'fields': [{'metadata_field': 'figure_title',...",NaN
1,054_Sudan-Basic-Education-Emergency-Support-Pr...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""figure_title"",""o...","{'input_tokens': 13799, 'input_tokens_details'...","{'input_tokens': 13799, 'output_tokens': 1576,...",completed,NaN,"{'fields': [{'metadata_field': 'figure_title',...",NaN
2,134_PAD7340ARABIC00n0220020140Arabic01_figure_...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""figure_title"",""o...","{'input_tokens': 13286, 'input_tokens_details'...","{'input_tokens': 13286, 'output_tokens': 1460,...",completed,NaN,"{'fields': [{'metadata_field': 'figure_title',...",NaN
3,004_BOSIB-87c444de-4797-4bf9-b654-4932a7fb0112...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""figure_title"",""o...","{'input_tokens': 8299, 'input_tokens_details':...","{'input_tokens': 8299, 'output_tokens': 2116, ...",completed,NaN,"{'fields': [{'metadata_field': 'figure_title',...",NaN
4,202_multi0page_figure_000.png,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""geographic_scope...","{'input_tokens': 12161, 'input_tokens_details'...","{'input_tokens': 12161, 'output_tokens': 2134,...",completed,NaN,{'fields': [{'metadata_field': 'geographic_sco...,NaN
...,...,...,...,...,...,...,...,...,...,...
205,document_8886604_table_005.png,prwp,gpt-5.5,"{""fields"":[{""metadata_field"":""table_title"",""ob...","{'input_tokens': 12587, 'input_tokens_details'...","{'input_tokens': 12587, 'output_tokens': 1475,...",completed,NaN,"{'fields': [{'metadata_field': 'table_title', ...",NaN
206,document_6284707_table_004.png,prwp,gpt-5.5,"{""fields"":[{""metadata_field"":""table_panel_titl...","{'input_tokens': 8238, 'input_tokens_details':...","{'input_tokens': 8238, 'output_tokens': 3145, ...",completed,NaN,{'fields': [{'metadata_field': 'table_panel_ti...,NaN
207,document_7095002_table_002.png,prwp,gpt-5.5,"{""fields"":[{""metadata_field"":""table_title"",""ob...","{'input_tokens': 9890, 'input_tokens_details':...","{'input_tokens': 9890, 'output_tokens': 1640, ...",completed,NaN,"{'fields': [{'metadata_field': 'table_title', ...",NaN
208,document_7063025_table_001.png,prwp,gpt-5.5,"{""fields"":[{""metadata_field"":""table_title"",""ob...","{'input_tokens': 8278, 'input_tokens_details':...","{'input_tokens': 8278, 'output_tokens': 1525, ...",completed,NaN,"{'fields': [{'metadata_field': 'table_title', ...",NaN


In [4]:
data[data["error"].notna()]

,snapshot_id,source,model,raw_output,usage,cost,status,incomplete_details,parsed_output,error


In [5]:
data["snapshot_type"] = data["snapshot_id"].apply(lambda x: x.split("_")[-2])
data.groupby(["source", "snapshot_type"]).agg(
    count=("source", "count"),
)

count
source  snapshot_type       
prwp    figure            35
        table             35
refugee figure            35
        table             35
unhcr   figure            35
        table             35

# Preprocessing

In [6]:
pd.set_option('display.max_colwidth', None)

In [7]:
df = data[["snapshot_id", "snapshot_type", "source", "model", "parsed_output"]].copy()
df["parsed_output"] = df["parsed_output"].apply(
    lambda x: x.get("fields", []) if isinstance(x, dict) else []
)

df = df.explode("parsed_output").reset_index(drop=True)
df = df.join(pd.json_normalize(df["parsed_output"])).drop(columns="parsed_output")

df.head()

,snapshot_id,snapshot_type,source,model,metadata_field,observed_value,description,source_level,discovery_value,reasoning
0,027_Jordan-Emergency-Food-Security-Project_figure_001.png,figure,refugee,gpt-5.5,figure_title,Figure 2: Share of food expenditure in total expenditure p.c. (%),The title or caption identifying the figure and its subject.,snapshot,high,The figure title provides the most direct searchable description of the snapshot content.
1,027_Jordan-Emergency-Food-Security-Project_figure_001.png,figure,refugee,gpt-5.5,indicator_name,Share of food expenditure in total expenditure p.c.,The main measure or indicator represented in the figure.,snapshot,high,Indicator-level metadata enables retrieval of figures by the specific economic or food-security measure shown.
2,027_Jordan-Emergency-Food-Security-Project_figure_001.png,figure,refugee,gpt-5.5,subject_domain,Food expenditure; food security,The thematic domain or topical area represented by the snapshot.,both,high,The figure title and document metadata both connect the snapshot to food expenditure and food security topics.
3,027_Jordan-Emergency-Food-Security-Project_figure_001.png,figure,refugee,gpt-5.5,geographic_scope,"Jordan, inferred from parent document title and project name; not visible in snapshot","The country, region, or geographic area to which the figure data applies.",document,high,Geographic context is essential for discovery because it is not shown in the figure itself.
4,027_Jordan-Emergency-Food-Security-Project_figure_001.png,figure,refugee,gpt-5.5,time_period,Not identifiable from this snapshot,"The date, year, or period covered by the data in the figure.",snapshot,high,"A data time period would be highly useful for comparison, but it is not visible in the figure."


In [8]:
# df.to_csv("discovery_results.csv", index=False)

# Source citation

In [9]:
source_df = df[df["metadata_field"].str.contains("source")].copy()
excl_list = [
    x
    for x in source_df["metadata_field"].unique()
    if any(sub in x for sub in ["financing", "funding"])
]
source_df = source_df[~source_df["metadata_field"].isin(excl_list)]
source_df

,snapshot_id,snapshot_type,source,model,metadata_field,observed_value,description,source_level,discovery_value,reasoning
12,027_Jordan-Emergency-Food-Security-Project_figure_001.png,figure,refugee,gpt-5.5,source_citation,Source: World Bank,The cited source or producer of the data shown in the snapshot.,both,high,Source attribution supports provenance tracking and trust assessment.
27,054_Sudan-Basic-Education-Emergency-Support-Project_figure_001.png,figure,refugee,gpt-5.5,source_citation,"Education Sector Analysis, 2018; http://www.earlygradereadingbarometer.org/",Data source or citation shown for the figure or its panels.,snapshot,high,Source citations provide provenance for the figure's underlying data.
38,134_PAD7340ARABIC00n0220020140Arabic01_figure_000.png,figure,refugee,gpt-5.5,data_source_name,دراسة استقصائية لمناخ الاستثمار / Investment Climate Survey,The named source from which the figure data were derived.,snapshot,high,Data source information supports provenance tracking and assessment of reliability.
39,134_PAD7340ARABIC00n0220020140Arabic01_figure_000.png,figure,refugee,gpt-5.5,source_publication_year,2012,The year associated with the cited data source.,snapshot,medium,Source year helps distinguish the data reference period from the publication or figure reference year.
40,134_PAD7340ARABIC00n0220020140Arabic01_figure_000.png,figure,refugee,gpt-5.5,source_citation,المصدر: دراسة استقصائية لمناخ الاستثمار (2012). / Source: Investment Climate Survey (2012).,The full source note or citation shown with the figure.,snapshot,high,"A full citation improves provenance, verification, and reuse of figure data."
...,...,...,...,...,...,...,...,...,...,...
2890,document_717473_table_019.png,table,prwp,gpt-5.5,data_source_type,"Household survey data, inferred from document abstract and opinion-response format",Type of source data underlying the table.,both,medium,Source-data type is useful for evaluating evidence quality and analytical reuse.
2905,document_9087385_table_002.png,table,prwp,gpt-5.5,source_citation,http://www.unodc.org/pdf/WDR_2005/volume_1_web.pdf,Source cited for the data shown in the table.,snapshot,high,The cited source provides provenance for validating and tracing the table data.
2932,document_11308693_table_006.png,table,prwp,gpt-5.5,data_source_name,LSIA; IRSHS; DREES; SOEP; NIDI; IADB; LKI; ENI; NIS; Pew,Named surveys or datasets used for the estimates in the table.,snapshot,high,Dataset names are valuable provenance and discovery metadata for analytical reuse.
2965,document_15151160_table_000.png,table,prwp,gpt-5.5,source_citation,Authors’ calculations based on UNCTAD TRAINS and Chemingui & Dessus (2008).,The cited data sources or references underlying the table.,snapshot,high,Source citations support provenance tracking and evaluation of data reliability.


In [10]:
source_df["metadata_field"].value_counts()

metadata_field
source_citation                    81
source_organization                12
data_source                        10
data_source_name                    9
source_document_title               7
data_source_citation                3
cartographic_source                 1
source_disclaimer                   1
data_source_or_responsible_unit     1
source_publication_year             1
institutional_source                1
data_source_methodology             1
source_access_date                  1
source_language                     1
source_url                          1
data_source_coverage                1
resource_or_service                 1
source_or_reference_note            1
data_source_year                    1
underlying_data_sources             1
data_source_abbreviation            1
source_datasets                     1
source_dataset                      1
source_document_type                1
data_sources                        1
data_source_type                   

In [11]:
df_to_clipboard(source_df["metadata_field"].value_counts(), index=True)

In [12]:
source_df["observed_value"].value_counts()

observed_value
Not identifiable from this snapshot                                                   32
World Bank                                                                            10
Development Economics                                                                  4
Authors’ calculations                                                                  3
Are commodity prices more volatile now ? a long-run perspective                        2
                                                                                      ..
Household survey data, inferred from document abstract and opinion-response format     1
http://www.unodc.org/pdf/WDR_2005/volume_1_web.pdf                                     1
LSIA; IRSHS; DREES; SOEP; NIDI; IADB; LKI; ENI; NIS; Pew                               1
Authors’ calculations based on UNCTAD TRAINS and Chemingui & Dessus (2008).            1
Authors’ estimation                                                                    1
Name: 

In [13]:
source_df[source_df["observed_value"] == "Not identifiable from this snapshot"]

,snapshot_id,snapshot_type,source,model,metadata_field,observed_value,description,source_level,discovery_value,reasoning
101,200_multi-page_table_003.png,table,refugee,gpt-5.5,source_citation,Not identifiable from this snapshot,Citation or source note identifying where the table data originated.,snapshot,medium,Source citation is valuable for provenance tracking even though no explicit citation is visible here.
146,120_PAD1205-ARABIC-PAD-PP152646-PUBLIC-Box393206B_table_008.png,table,refugee,gpt-5.5,source_citation,Not identifiable from this snapshot,The data source or citation associated with the table.,snapshot,medium,"Source citation is important for provenance and validation, but no source is visible in the snapshot."
385,121_PAD1190-PAD-P152848-PUBLIC-Box391435B-LB-EESSP-Final-PAD-for-printing_figure_001.png,figure,refugee,gpt-5.5,source_citation,Not identifiable from this snapshot,The cited data source or provenance statement for the figure's data.,snapshot,medium,Source citation is important for provenance even when it is absent from the visible figure.
490,121_PAD1190-PAD-P152848-PUBLIC-Box391435B-LB-EESSP-Final-PAD-for-printing_figure_000.png,figure,refugee,gpt-5.5,source_citation,Not identifiable from this snapshot,The cited source or origin of the data used in the figure.,snapshot,medium,Source citation is important for provenance tracking even though it is not visible in this figure.
1055,white_paper_social_media_3_0_figure_003.png,figure,unhcr,gpt-5.5,source_citation,Not identifiable from this snapshot,Citation or source statement for the data used in the figure.,snapshot,medium,"A source citation would be important for provenance tracking, but none is visible in the snapshot."
1068,rapport_mensuel_monitoring_de_protection_de_la_region_de_maradi_au_niger_octobre_2022_figure_005.png,figure,unhcr,gpt-5.5,source_citation,Not identifiable from this snapshot,Source or provenance citation for the data shown in the figure.,snapshot,medium,"A data source citation would support provenance tracking, but none is visible."
1082,refugee_and_migrant_children_in_europe_-_accompanied_unaccompanied_and_separated_-_overview_of_trends_january_-_december_2020_figure_007.png,figure,unhcr,gpt-5.5,source_citation,Not identifiable from this snapshot,The cited data source or provenance statement for the figure.,snapshot,medium,Source citation supports provenance tracking and assessment of data reliability.
1111,rapport_du_monitoring_de_protection_de_la_region_de_diffa_au_niger_janvier_2022_figure_004.png,figure,unhcr,gpt-5.5,source_citation,Not identifiable from this snapshot,The cited source or provenance for the data shown in the figure.,snapshot,medium,"Source information supports provenance tracking and trust assessment, but no citation is visible."
1124,08072015_northgovernorateprofile_table_000.png,table,unhcr,gpt-5.5,source_citation,Not identifiable from this snapshot,Source organization or citation for the data shown in the figure.,snapshot,high,A source citation is important for provenance and trust but is not visible in the snapshot.
1227,echoes_en_8_figure_000.png,figure,unhcr,gpt-5.5,source_citation,Not identifiable from this snapshot,Source or provenance statement for the data shown in the figure.,snapshot,medium,Source citation is important for provenance tracking even when it is absent from the visible snapshot.


# Top metadata fields

In [14]:
df["metadata_field"].value_counts().head(10)

metadata_field
geographic_scope      192
unit_of_measure       180
indicator_name        152
time_period           149
row_dimension         102
visualization_type     99
table_title            90
column_dimension       88
figure_title           84
population_group       84
Name: count, dtype: int64

# Geographic scope

In [15]:
geo = df[df["metadata_field"] == "geographic_scope"].copy()

In [16]:
geo["snapshot_id"].nunique()

192

In [17]:
geo["source_level"].value_counts()

source_level
both        71
snapshot    65
document    56
Name: count, dtype: int64

# Population group

In [18]:
pop = df[df["metadata_field"] == "population_group"].copy()

In [19]:
pop["snapshot_id"].nunique()

84